In [1]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import VotingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score


import matplotlib.pyplot as plt
import utils_, config, model
import os
import numpy as np
import pandas as pd
import Analysis_function
import preprocess

import warnings
warnings.filterwarnings('ignore')


def train_with_gridsearch(X, y, verbose=True):
    """Train XGBoost with GridSearchCV."""
    param_grid = {
        'max_depth': [3, 5],
        'n_estimators': [50, 100],
        'learning_rate': [0.05, 0.1]
    }

    best_params, best_model = manual_gridsearch(X, y, param_grid)

    return best_params, best_model


from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

def manual_gridsearch(X, y, param_grid):
    best_model = None
    best_score = -1
    best_params = {}

    for depth in param_grid['max_depth']:
        for n_estimators in param_grid['n_estimators']:
            for lr in param_grid['learning_rate']:
                model = XGBClassifier(
                    max_depth=depth,
                    n_estimators=n_estimators,
                    learning_rate=lr,
                    use_label_encoder=False,
                    eval_metric='logloss',
                    random_state=42
                )

                X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
                model.fit(X_train, y_train)

                y_pred = model.predict(X_test)
                acc = accuracy_score(y_test, y_pred)

                if acc > best_score:
                    best_score = acc
                    best_model = model
                    best_params = {'max_depth': depth, 'n_estimators': n_estimators, 'learning_rate': lr}

    print("Best Params:", best_params)
    print("Best Accuracy:", best_score)

    return best_params, best_model


def stack_and_train(X, y, model_outputs, verbose=True):
    """Add model output features and train with tuned XGBoost."""
    for name, pred in model_outputs.items():
        X[name] = pred
    best_param, best_model = train_with_gridsearch(X, y, verbose)
    acc = accuracy_score(y, best_model.predict(X))
    return acc, best_model, X


C:\Users\hml76\PycharmProjects\HRD2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. Fetch dataset in dict format - Work data / Head data
dataset_Work = preprocess.get_all_data(config.year_list, config.file_names_Work)
dataset_Head = preprocess.get_all_data(config.year_list, config.file_names_Head)

# 2. Check the every column and its semantic name
#utils_.see_col_idx_and_name(dataset['2020']['data'], dataset['2020']['meta'])

# 3. Check the intersection for the number of organization - year by year (e.g. compare 2020 - 2021)
#_ = Analysis_function.compare_company_ids_in_dataset(dataset_Work)

# 4. Check the intersection for the number of organization - All years (2020-2023) - we do this again in next step
#_ = Analysis_function.get_common_company_ids_all_years(dataset_Work)

# 5. Check the intersection for the number of organization (All) and filter; select only the organization that involves throughout all years
dataset_Work = preprocess.filter_dataset_by_common_ids(dataset_Work)

# 6. Target variable check - 1. Type count (value_counts()) / 2. check Nan
Analysis_function.target_variable_check(dataset_Work, target_variable=config.target_col)

# 7. Nan 값 25% 언더면 정수형 평균값으로 넣고, 위면 해당 컬럼 삭제
dataset_Work = preprocess.clean_all_years(dataset_Work, columns_to_drop=[], verbose=True)

# 8. Store only the common columns in each yearly dataset - 컬럼이 다르면 안되니깐.
dataset_Work = preprocess.unify_columns_by_base_name(dataset_Work)

HCCP_2ndWave_Work_1st_v2.sav ===> 2020 data
HCCP_2ndWave_Work_2nd_v2.sav ===> 2021 data
HCCP_2ndWave_Work_3rd_v3.sav ===> 2022 data
HCCP_2ndWave_Work_4th.sav ===> 2023 data
HCCP_2ndWave_Head_1st(최종).sav ===> 2020 data
HCCP_2ndWave_Head_2nd(최종).sav ===> 2021 data
HCCP_2ndWave_Head_3rd(최종).sav ===> 2022 data
HCCP_2ndWave_Head_4th.sav ===> 2023 data
공통 기업 ID 개수: 384
2020 필터링 후 행 개수: 7054
2021 필터링 후 행 개수: 7613
2022 필터링 후 행 개수: 7338
2023 필터링 후 행 개수: 8740
2020 - W20Q09A : NaN 0개 / 전체 7054개 (0.00%)
W20Q09A
3.0    2372
8.0    1682
4.0    1534
2.0     873
5.0     434
1.0     159
Name: count, dtype: int64
2021 - W21Q09A : NaN 0개 / 전체 7613개 (0.00%)
W21Q09A
-8.0    2331
 3.0    2318
 4.0    1511
 2.0     907
 5.0     369
 1.0     177
Name: count, dtype: int64
2022 - W22Q09A : NaN 0개 / 전체 7338개 (0.00%)
W22Q09A
3.0    2263
8.0    1922
4.0    1436
2.0     893
5.0     501
1.0     323
Name: count, dtype: int64
2023 - W23Q09A : NaN 0개 / 전체 8740개 (0.00%)
W23Q09A
3.0    4008
4.0    2034
2.0    1133
5.0   

In [3]:

ACC1 = {}
models_dict = {}

for year in ['2020', '2021', '2022', '2023']:

    # --------------- YEAR 2020 ---------------
    if year == '2020':
        for target_year in ['2021', '2023']:
            df, label_col = preprocess.merge_worker_head_labels(dataset_Work, dataset_Head, year_w=year, year_h=target_year)
            df = preprocess.standardize_feature_names(df)
            X, y = preprocess.clean_target_classes(df, target_col=label_col)
            if target_year == '2021':
                X_20_21 = X
            if target_year == '2023':
                X_20_23 = X
            acc, XGB_model = train_with_gridsearch(X, y)#, None
            models_dict[f"model20_{target_year[2:]}"] = XGB_model
            print(f"[{year}→{target_year}]") # Accuracy: {acc:.4f}")
        ACC1[year] = acc

    # --------------- YEAR 2021 ---------------
    elif year == '2021':
        # For 2021 ➝ 2021
        df, label_col = preprocess.merge_worker_head_labels(dataset_Work, dataset_Head, year_w='2021', year_h='2021')
        df = preprocess.standardize_feature_names(df)
        X, y = preprocess.clean_target_classes(df, target_col=label_col)

        X_stack = X.copy()
        X_stack['model20_21_output'] = models_dict['model20_21'].predict_proba(X)[:, 1]
        acc, model_21_21, _ = stack_and_train(X_stack, y, {}, verbose=False)

        # For 2021 ➝ 2023
        df, label_col = preprocess.merge_worker_head_labels(dataset_Work, dataset_Head, year_w='2021', year_h='2023')
        df = preprocess.standardize_feature_names(df)
        X, y = preprocess.clean_target_classes(df, target_col=label_col)

        X_stack = X.copy()
        X_stack['model20_23_output'] = models_dict['model20_23'].predict_proba(X)[:, 1]
        acc2, model_21_23, _ = stack_and_train(X_stack, y, {}, verbose=False)

        models_dict['model21_21'] = model_21_21
        models_dict['model21_23'] = model_21_23
        ACC1[year] = [acc, acc2]

    # --------------- YEAR 2022 ---------------
    elif year == '2022':
        df, label_col = preprocess.merge_worker_head_labels(dataset_Work, dataset_Head, year_w='2022', year_h='2023')
        df = preprocess.standardize_feature_names(df)
        X, y = preprocess.clean_target_classes(df, target_col=label_col)

        outputs = {
            'model20_21_output': models_dict['model20_21'].predict_proba(X)[:, 1],
            'model20_23_output': models_dict['model20_23'].predict_proba(X)[:, 1],
        }
        output_for_next = outputs.copy()

        X_tmp = X.copy()
        X_tmp['model20_21_output'] = outputs['model20_21_output']
        outputs['model21_21_output'] = models_dict['model21_21'].predict_proba(X_tmp)[:, 1]
        X_21_21 = X_tmp.copy()

        X_tmp = X.copy()
        X_tmp['model20_23_output'] = outputs['model20_23_output']
        outputs['model21_23_output'] = models_dict['model21_23'].predict_proba(X_tmp)[:, 1]
        X_21_23 = X_tmp.copy()

        acc, model_22_23, X_22_23 = stack_and_train(X, y, outputs)
        models_dict['model22_23'] = model_22_23
        ACC1[year] = acc

    # --------------- YEAR 2023 ---------------
    elif year == '2023':
        df, label_col = preprocess.merge_worker_head_labels(dataset_Work, dataset_Head, year_w='2023', year_h='2023')
        df = preprocess.standardize_feature_names(df)
        X, y = preprocess.clean_target_classes(df, target_col=label_col)

        X_21_21 = X.copy()
        X_21_21['model20_21_output'] = models_dict['model20_21'].predict_proba(X)[:, 1]
        #X_21_21['model21_21_output'] = models_dict['model21_21'].predict_proba(X_21_21)[:, 1]

        X_21_23 = X.copy()
        X_21_23['model20_23_output'] = models_dict['model20_23'].predict_proba(X)[:, 1]
        #X_21_23['model21_23_output'] = models_dict['model21_23'].predict_proba(X_21_23)[:, 1]

        X_22_23_v1, X_22_23_v2 = X.copy(), X.copy()
        X_22_23_v1['model20_21_output'] = models_dict['model20_21'].predict_proba(X)[:, 1]
        X_22_23_v2['model20_23_output'] = models_dict['model20_23'].predict_proba(X)[:, 1]

        tmp_21_21 = models_dict['model21_21'].predict_proba(X_22_23_v1)[:, 1]
        tmp_21_23 = models_dict['model21_23'].predict_proba(X_22_23_v2)[:, 1]

        X_22_23_v1['model20_23_output'] = models_dict['model20_23'].predict_proba(X)[:, 1]
        X_22_23_v1['model21_21_output'] = tmp_21_21
        X_22_23_v1['model21_23_output'] = tmp_21_23

        #X_22_23_v1['model22_23_output'] = models_dict['model22_23'].predict_proba(X_22_23_v1)[:, 1]


        outputs = {
            'model20_21_output': models_dict['model20_21'].predict_proba(X)[:, 1],
            'model20_23_output': models_dict['model20_23'].predict_proba(X)[:, 1],
            'model21_21_output': models_dict['model21_21'].predict_proba(X_21_21)[:, 1],
            'model21_23_output': models_dict['model21_23'].predict_proba(X_21_23)[:, 1],
            'model22_23_output': models_dict['model22_23'].predict_proba(X_22_23_v1)[:, 1],
        }

        acc, _, _ = stack_and_train(X_22_23_v1, y, outputs)
        ACC1[year] = acc

Year: 2020 | Merged shape: (7054, 109) | work shape: (7054, 126) | head shape: (500, 2)
Best Params: {'max_depth': 3, 'n_estimators': 100, 'learning_rate': 0.1}
Best Accuracy: 0.765625
[2020→2021]
Year: 2020 | Merged shape: (7054, 109) | work shape: (7054, 126) | head shape: (500, 2)
Best Params: {'max_depth': 3, 'n_estimators': 100, 'learning_rate': 0.1}
Best Accuracy: 0.8911764705882353
[2020→2023]
Year: 2021 | Merged shape: (7613, 109) | work shape: (7613, 126) | head shape: (500, 2)
Best Params: {'max_depth': 3, 'n_estimators': 100, 'learning_rate': 0.05}
Best Accuracy: 0.7537688442211056
Year: 2021 | Merged shape: (7613, 109) | work shape: (7613, 126) | head shape: (500, 2)
Best Params: {'max_depth': 3, 'n_estimators': 100, 'learning_rate': 0.05}
Best Accuracy: 0.8810975609756098
Year: 2022 | Merged shape: (7338, 109) | work shape: (7338, 126) | head shape: (500, 2)
Best Params: {'max_depth': 3, 'n_estimators': 50, 'learning_rate': 0.05}
Best Accuracy: 0.9252336448598131
Year: 202

In [4]:
ACC1

{'2020': {'max_depth': 3, 'n_estimators': 100, 'learning_rate': 0.1},
 '2021': [0.81649069884364, 0.876219512195122],
 '2022': 0.9201995012468828,
 '2023': 0.9517669531996179}

In [8]:
X_22_23_v1#['Q18B']

,Q05A,Q14E,Q061C,Q061A,Q16B1,Q26C,Q081D,Q26B,Q14A,Q20,...,Q14C,Q22F,Q081B,Q22D,Q01A,model20_21_output,model20_23_output,model21_21_output,model21_23_output,model22_23_output
0,2018.0,3.0,2.0,2.0,2.0,4.0,3.0,2.0,3.0,3.0,...,2.0,4.0,2.0,3.0,2012.0,0.770347,0.861837,0.684937,0.857594,0.951244
1,2017.0,4.0,4.0,1.0,2.0,4.0,3.0,3.0,4.0,3.0,...,4.0,4.0,3.0,4.0,2015.0,0.892436,0.912672,0.895096,0.827960,0.988404
2,2013.0,3.0,4.0,3.0,2.0,3.0,2.0,2.0,3.0,3.0,...,2.0,3.0,2.0,3.0,2013.0,0.749667,0.725704,0.454275,0.596094,0.225334
3,2015.0,2.0,3.0,3.0,2.0,4.0,2.0,2.0,2.0,2.0,...,1.0,3.0,1.0,3.0,2015.0,0.739019,0.973060,0.422796,0.914208,0.994003
4,2022.0,3.0,3.0,2.0,2.0,2.0,2.0,1.0,4.0,2.0,...,2.0,2.0,2.0,2.0,2010.0,0.701897,0.856970,0.412242,0.840490,0.666644
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2089,2006.0,2.0,3.0,3.0,2.0,3.0,2.0,3.0,3.0,1.0,...,3.0,2.0,3.0,3.0,2006.0,0.321842,0.351240,0.134216,0.100154,0.033473
2090,2014.0,3.0,2.0,3.0,2.0,3.0,3.0,3.0,2.0,1.0,...,3.0,2.0,3.0,3.0,2014.0,0.550848,0.955049,0.389708,0.859142,0.976483
2091,2009.0,4.0,2.0,3.0,2.0,3.0,3.0,3.0,3.0,1.0,...,3.0,2.0,3.0,3.0,2009.0,0.521507,0.278545,0.158207,0.029938,0.012049
2092,2020.0,3.0,2.0,3.0,2.0,3.0,2.0,3.0,3.0,1.0,...,2.0,2.0,3.0,3.0,2020.0,0.519003,0.715751,0.303959,0.432775,0.268132
